<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 2: Case B and Constraint Structure

**The advertising plan is acceptable only when both individual spending limits and the shared budget requirement hold.**

Part 1 introduced the standard symbols through the unconstrained clock case. This part formulates the advertising case from 01-2 and uses it to distinguish bound-constrained and generally constrained problems.

### 1 · Carry forward the advertising decision

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_b_advertising.png" alt="One advertising budget divided between social-media announcements on a phone and printed flyers." width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Case B from 01-2: social-media spending \(s\) and flyer spending \(r\) form \(x=[s,r]^{\mathsf T}\). The shared budget couples the decisions.

The club chooses social-media spending \(s\) and flyer spending \(r\), both in dollars. The forecast reach is \(R(s,r)=4s+2r\). The requirements are

> $\displaystyle s+r\le100,\qquad 0\le s\le70,\qquad 10\le r\le100.$

The spending changes the real advertising plan. Forecast reach and total spending are produced quantities.

### 2 · Translate every requirement into the standard form

Let

> $\displaystyle x=\begin{bmatrix}s\\r\end{bmatrix},\qquad
y=\operatorname{Sim}(x)=\begin{bmatrix}R(x)\\B(x)\end{bmatrix}
=\begin{bmatrix}4s+2r\\s+r\end{bmatrix}.$

Because the shared template minimizes a scalar objective, maximizing reach is written as minimizing its negative:

> $\displaystyle f(y)=-R(x).$

Write every inequality as a residual that must be nonpositive:

| Requirement | Residual form |
|:---|:---|
| \(s\ge0\) | \(g_1(x)=-s\le0\) |
| \(s\le70\) | \(g_2(x)=s-70\le0\) |
| \(r\ge10\) | \(g_3(x)=10-r\le0\) |
| \(r\le100\) | \(g_4(x)=r-100\le0\) |
| \(s+r\le100\) | \(g_5(x)=s+r-100\le0\) |

There are \(m_g=5\) inequality constraints and \(m_h=0\) equality constraints. A negative residual has slack, zero is active, and a positive residual is a violation.

The complete formulation is

> $\displaystyle \underset{x\in\mathbb R^2}{\operatorname{minimize}}\quad -(4s+2r)$
>
> $\displaystyle \text{subject to}\quad g_j(x)\le0\quad(j=1,\ldots,5).$

### 3 · Evaluate feasibility before reach

The left panel places \((s,r)\) in the spending plane. Teal shading marks feasible plans, and the numbered blue lines connect plans with equal forecast reach \(R\). The right panel shows all five inequality residuals.

The orange plan \((70,40)\) lies beyond the total-budget boundary. Its budget residual is \(10>0\), so its reach cannot make it eligible. The gold star marks the best feasible plan on the 1-dollar search grid.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/case_b_formulation_graph.png" alt="A spending-space feasible region and constraint-residual bars connect an advertising plan to its reach and budget violation." width="1000" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
import numpy as np

# Baseline requirement limit
TOTAL_BUDGET = 100.0


def simulate_advertising(x):
    social, flyers = np.asarray(x, dtype=float)
    return {"reach": 4.0 * social + 2.0 * flyers, "spending": social + flyers}


def objective_function(y):
    return -float(y["reach"])


def inequality_constraints(x, y, budget_limit=TOTAL_BUDGET):
    social, flyers = np.asarray(x, dtype=float)
    return np.array([
        -social,
        social - 70.0,
        10.0 - flyers,
        flyers - 100.0,
        y["spending"] - float(budget_limit),
    ])


def evaluate_candidate(x, budget_limit=TOTAL_BUDGET):
    decision = np.asarray(x, dtype=float)
    response = simulate_advertising(decision)
    residuals = inequality_constraints(decision, response, budget_limit)
    return {
        "x": decision,
        "y": response,
        "f": objective_function(response),
        "g": residuals,
        "feasible": bool(np.all(residuals <= 1e-10)),
    }

In [ ]:
import sys
import matplotlib


def _pyplot(*, interactive=False):
    """Use the course's widget-backend fallback outside the browser runtime."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
PURPLE = "#8856a7"
GRAY = "#a6a6a6"
GOLD = "#f6c945"


def case_canvas(title, controls, *, panels=2, interactive=False):
    """Create a shared layout; controls are (name, label, low, high, value, step, color)."""
    plt = _pyplot(interactive=interactive)
    figure, axes = plt.subplots(
        1, panels, figsize=(12.4 if panels == 2 else 14.0, 7.4 if interactive else 5.3)
    )
    figure.suptitle(title, y=0.985, fontsize=14, fontweight="bold")
    status = figure.text(0.5, 0.918, "", ha="center", va="center", fontsize=11)
    footer = figure.text(0.5, 0.045 if not interactive else 0.27, "",
                         ha="center", va="center", fontsize=9)
    sliders = {}
    if interactive:
        from matplotlib.widgets import Slider

        figure.subplots_adjust(left=0.075, right=0.97,
                               bottom=0.41 if panels == 3 else 0.37, top=0.83, wspace=0.38)
        positions = np.linspace(0.19, 0.065, max(len(controls), 2))
        for position, (name, label, low, high, value, step, color) in zip(positions, controls):
            slider_axis = figure.add_axes([0.28, position, 0.61, 0.026])
            sliders[name] = Slider(slider_axis, label, low, high, valinit=value,
                                   valstep=step, valfmt="%1.0f" if step >= 1 else "%1.2f",
                                   color=color, initcolor=color)
    figure._case_sliders = sliders
    figure._case_state = {}
    return plt, figure, axes, sliders, status, footer


def finish_case(plt, figure, axes, sliders, refresh, *, interactive=False):
    """Connect controls and keep widgets and evaluated records alive on the figure."""
    for axis in axes:
        axis.grid(alpha=0.25)
    for slider in sliders.values():
        slider.on_changed(refresh)
    figure._case_refresh = refresh
    refresh()
    if not interactive:
        figure.tight_layout(rect=(0.015, 0.09, 0.985, 0.96))
    plt.show()
    if not interactive:
        plt.close(figure)
    return figure


def show_advertising_formulation(decision=(70.0, 40.0), budget_limit=100.0, *, interactive=False):
    controls = [
        ("social", "Decision: social spending s ($)", 0.0, 100.0, decision[0], 1.0, BLUE),
        ("flyers", "Decision: flyer spending r ($)", 0.0, 100.0, decision[1], 1.0, BLUE),
        ("budget", "Requirement: total budget ($)", 60.0, 140.0, budget_limit, 1.0, TEAL),
    ]
    plt, figure, axes, sliders, status, footer = case_canvas(
        "Case B · High reach is useful only inside the feasible region",
        controls, interactive=interactive,
    )
    from matplotlib.patches import Polygon

    axes[0].set_facecolor("#f0f0f0")
    feasible_patch = Polygon([(0, 10), (70, 10), (70, 30), (0, 100)],
                              color=TEAL, alpha=0.18, label="Feasible region")
    axes[0].add_patch(feasible_patch)
    mesh = np.linspace(0, 100, 101)
    social, flyers = np.meshgrid(mesh, mesh)
    contours = axes[0].contour(social, flyers, 4 * social + 2 * flyers,
                               levels=[100, 200, 300, 340, 400, 500],
                               colors=BLUE, linewidths=0.8, alpha=0.75)
    axes[0].clabel(contours, fontsize=8, fmt="%d")
    budget_line, = axes[0].plot([0, 100], [budget_limit, budget_limit - 100], "--",
                                color="#555555", label="Total-budget boundary")
    axes[0].plot([70, 70], [0, 100], ":", color="#555555", linewidth=1)
    axes[0].plot([0, 100], [10, 10], ":", color="#555555", linewidth=1)
    axes[0].plot([0, 100], [100, 100], ":", color="#555555", linewidth=1)
    records = [evaluate_candidate([s, r]) for s in range(101) for r in range(101)]
    bounded_records = [r for r in records if np.all(r["g"][:4] <= 1e-10)]
    best_marker = axes[0].scatter([], [], marker="*", color=GOLD, edgecolor="black", s=230,
                                   zorder=5, label="Best $1-grid candidate")
    current_marker = axes[0].scatter([], [], color=ORANGE, edgecolor="black", s=105,
                                     zorder=6, label="Current plan")
    axes[0].set(xlabel="Social-media spending s ($)", ylabel="Flyer spending r ($)",
                xlim=(-4, 104), ylim=(-4, 140), title="Blue lines: equal forecast reach R")
    axes[0].legend(fontsize=8, loc="upper right")
    names = ["Social lower", "Social upper", "Flyer lower", "Flyer upper", "Total budget"]
    residual_bars = axes[1].barh(names, np.zeros(5), color=TEAL)
    axes[1].invert_yaxis()
    axes[1].axvline(0, color="black", linewidth=1.3)
    annotations = [axes[1].text(0, i, "", va="center", fontsize=10) for i in range(5)]
    axes[1].set(xlabel="Inequality residual g ($)", xlim=(-112, 115),
                title="Every residual must be at or below zero")
    footer.set_text("Teal residuals pass; gray residuals violate a requirement. Positive g rejects a plan regardless of reach.")

    def refresh(_=None):
        values = (sliders["social"].val, sliders["flyers"].val) if sliders else decision
        budget = sliders["budget"].val if sliders else budget_limit
        current = evaluate_candidate(values, budget_limit=budget)
        selected = min((r for r in bounded_records if r["y"]["spending"] <= budget),
                        key=lambda r: r["f"])
        best = evaluate_candidate(selected["x"], budget_limit=budget)
        best_marker.set_offsets([best["x"]])
        right_bound = min(70.0, budget - 10.0)
        top_edge = [(s, min(100.0, budget - s)) for s in np.linspace(right_bound, 0, 71)]
        feasible_patch.set_xy([(0, 10), (right_bound, 10), *top_edge])
        budget_line.set_ydata([budget, budget - 100])
        current_marker.set_offsets([current["x"]])
        axes[1].set_xlim(min(-112, float(current["g"].min()) - 25),
                         max(115, float(current["g"].max()) + 25))
        for bar, label, residual in zip(residual_bars, annotations, current["g"]):
            bar.set_width(residual)
            bar.set_color(TEAL if residual <= 1e-10 else GRAY)
            label.set_x(residual + (3 if residual >= 0 else -3))
            label.set_ha("left" if residual >= 0 else "right")
            label.set_text(f"{residual:.0f}")
        eligibility = "FEASIBLE" if current["feasible"] else "REJECTED"
        status.set_text(f"Current (s, r) = ({values[0]:.0f}, {values[1]:.0f})"
                        f"   |   Reach R = {current['y']['reach']:.0f}"
                        f"   |   f = -R = {current['f']:.0f}   |   Budget = {budget:.0f}   |   {eligibility}")
        status.set_color(TEAL if current["feasible"] else "#555555")
        figure._case_state.update(current=current, best=best, budget_limit=budget)
        figure.canvas.draw_idle()

    return finish_case(plt, figure, axes, sliders, refresh, interactive=interactive)

The two blue trackbars change spending \(s\) and \(r\). The teal trackbar changes the total-budget requirement.

For this controlled variation, replace the baseline limit 100 with \(B_{\max}\):

> $\displaystyle g_5(x;B_{\max})=s+r-B_{\max}\le0.$

The other four requirements and the reach forecast stay fixed. Moving \(B_{\max}\) shifts the budget boundary and can move the gold grid selection. For a fixed plan, it changes feasibility without changing reach.

In [ ]:
formulation_explorer = show_advertising_formulation(
    decision=(70.0, 40.0), interactive=True
)

The plan \((70,40)\) has greater reach than \((70,30)\), but its shared-budget residual is \(10>0\). It is rejected before objective values are compared. The reported result is the best candidate on the stated 1-dollar grid.

### 4 · Change only the restrictions

Keep the same decision and objective, then compare three formulations:

| Formulation | Restrictions retained | Classification |
|:---|:---|:---|
| A | None; \(x\in\mathbb R^2\) | Unconstrained |
| B | Only \(0\le s\le70\) and \(10\le r\le100\) | Bound-constrained |
| C | Bounds and \(s+r\le100\) | Generally constrained |

Formulation A is unbounded: forecast reach can increase without limit. “Unconstrained” names the restriction structure; it does not guarantee that a useful optimum exists.

Formulation C is generally constrained because \(s+r\le100\) couples the two decisions. An exact requirement would use an equality residual. For example, spending exactly 100 dollars is \(h_1(x)=s+r-100=0\).

### 5 · Classify the full Case B formulation

Case B is a **generally constrained, continuous, single-objective, linear, direct algebraic, deterministic optimization problem**. Both the objective and every constraint are linear in \(x\).

### Takeaway

Convert requirements before comparing candidates:

> **write bounds and residuals → check every \(g_j\le0\) and \(h_k=0\) → reject any violation → compare \(f\) only among feasible candidates**

Removing or adding a requirement changes the optimization formulation even when the real decision and response calculation stay the same. Part 3 changes the allowed value types instead.